## Import Libraries

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model selection
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score, roc_curve, auc

# Models
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from xgboost import XGBClassifier, XGBRegressor

Explanation:
This cell imports all required libraries for:

Data handling (pandas, numpy)
Visualization (matplotlib, seaborn)
Machine learning models and evaluation

# #2. Load Dataset

In [ ]:
df = pd.read_csv("/content/f1_clean_dataset_final.csv")
df.head()

Loads dataset and previews first rows.

Dataset Info

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

Checks size
Data types
Missing values

Data Cleaning

In [ ]:
df = df.dropna()

Removes missing values to ensure clean training.

Convert Time to Seconds

In [ ]:
def time_to_seconds(t):
    if isinstance(t, str):
        mins, rest = t.split(":")
        return float(mins) * 60 + float(rest)
    return t

df['q1'] = df['q1'].apply(time_to_seconds)
df['q2'] = df['q2'].apply(time_to_seconds)
df['q3'] = df['q3'].apply(time_to_seconds)

Converts lap times into numeric values → models need numbers.

Feature Selection

In [ ]:
features = ['grid', 'q1', 'q2', 'q3']

X = df[features]

y_class = df['top3']           # Classification target
y_reg = df['final_position']  # Regression target

Defines input features
Defines targets

Basic Statistics

In [ ]:
df.describe()

Mean
Std
Min/Max

Distribution of Target (top3)

In [ ]:
sns.countplot(x='top3', data=df)
plt.title("Distribution of Top 3 Finish")
plt.show()

Shows class balance.

0 = not top 3 (more)
1 = top 3 (less)



Distribution of Final Position

In [ ]:
plt.hist(df['final_position'], bins=20)
plt.title("Distribution of Final Position")
plt.xlabel("Position")
plt.ylabel("Frequency")
plt.show()

Shows how positions are spread.

Grid vs Final Position

In [ ]:
plt.scatter(df['grid'], df['final_position'])
plt.xlabel("Grid Position")
plt.ylabel("Final Position")
plt.title("Grid vs Final Position")
plt.show()

“Grid position shows a strong correlation with final race position.”

Correlation Heatmap

In [ ]:
corr = df[['grid', 'q1', 'q2', 'q3', 'final_position']].corr()

plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True)
plt.title("Correlation Matrix")
plt.show()

Shows relationships between variables.
q1, q2, q3 highly correlated
grid correlated with final_position

Boxplot (Outliers)

In [ ]:
sns.boxplot(data=df[['grid', 'q1', 'q2', 'q3']])
plt.title("Boxplot for Features")
plt.show()

Shows spread + outliers.

Top3 vs Grid

In [ ]:
sns.boxplot(x='top3', y='grid', data=df)
plt.title("Grid Position vs Top 3 Finish")
plt.show()

“Drivers starting in lower grid positions are more likely to finish in top 3.”

Train Test Split

In [ ]:
X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X, y_class, test_size=0.2, random_state=42
)

_, _, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

Splits data into:

Training (80%)
Testing (20%)

Classification Models

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

results_cls = []

for name, model in classifiers.items():
    model.fit(X_train, y_train_cls)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train_cls, train_pred)
    test_acc = accuracy_score(y_test_cls, test_pred)
    f1 = f1_score(y_test_cls, test_pred)

    results_cls.append((name, train_acc, test_acc, f1))

results_cls_df = pd.DataFrame(results_cls, columns=["Model", "Train Acc", "Test Acc", "F1 Score"])
results_cls_df

Trains multiple models
Compares performance
Includes overfitting check

Confusion Matrix

In [ ]:
best_model = RandomForestClassifier()
best_model.fit(X_train, y_train_cls)

y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test_cls, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(y_test_cls, y_pred))

Shows:

True vs predicted classes
Errors
Precision/Recall

ROC Curve

In [ ]:
y_prob = best_model.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test_cls, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

Evaluates classification performance beyond accuracy.

Regression Models

In [ ]:
regressors = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "XGBoost": XGBRegressor()
}

results_reg = []

for name, model in regressors.items():
    model.fit(X_train, y_train_reg)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    r2 = r2_score(y_test_reg, y_pred)

    results_reg.append((name, rmse, r2))

results_reg_df = pd.DataFrame(results_reg, columns=["Model", "RMSE", "R2"])
results_reg_df

Compares regression models using:

RMSE
R²

Actual vs Predicted

In [ ]:
plt.figure()
plt.scatter(y_test_reg, y_pred)
plt.xlabel("Actual Position")
plt.ylabel("Predicted Position")
plt.title("Actual vs Predicted")
plt.show()

Visualizes prediction accuracy.

Feature Importance

In [ ]:
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train_cls)

importances = rf_model.feature_importances_

plt.figure()
plt.barh(features, importances)
plt.title("Feature Importance")
plt.xlabel("Importance Score")
plt.show()

Shows which features matter most.

Cross Validation

In [ ]:
for name, model in classifiers.items():
    scores = cross_val_score(model, X, y_class, cv=5)
    print(f"{name}: {scores.mean():.3f}")

Ensures model stability across folds.

Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=3)
grid.fit(X_train, y_train_cls)

print("Best Params:", grid.best_params_)

Optimizes model performance.